# Daily Ecommerce Orders – ETL Pipeline Walkthrough

This notebook walks through each stage of the ETL pipeline step-by-step.

**Pipeline stages:**
1. **Extract** – load the raw CSV and validate schema
2. **Transform** – clean, type-cast, quality-check, and enrich
3. **Load** – persist to CSV files and a SQLite database
4. **Analysis** – explore the cleaned data with charts and SQL queries

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sqlite3

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded.')

---
## Stage 1 – Extract

In [ ]:
from etl.extract import extract
from etl import config

raw_df = extract(config.RAW_FILE)
print(f'Shape : {raw_df.shape}')
raw_df.head()

In [ ]:
null_summary = pd.DataFrame({
    'null_count': raw_df.isnull().sum(),
    'null_pct':   (raw_df.isnull().sum() / len(raw_df) * 100).round(2)
})
null_summary[null_summary['null_count'] > 0]

---
## Stage 2 – Transform

In [ ]:
from etl.transform import transform

clean_df, rejected_df = transform(raw_df)

print(f'Clean rows   : {len(clean_df):,}')
print(f'Rejected rows: {len(rejected_df):,}')
print(f'Rejection %  : {len(rejected_df)/len(raw_df)*100:.1f}%')
clean_df.head(3)

In [ ]:
if not rejected_df.empty:
    reason_counts = (
        rejected_df['rejection_reason']
        .str.split(';').explode()
        .loc[lambda s: s != '']
        .value_counts()
    )
    print('Rejection reason breakdown:')
    print(reason_counts.to_string())
else:
    print('No rows rejected.')

In [ ]:
eng_cols = [
    'order_id', 'order_date', 'order_year', 'order_month',
    'order_quarter_label', 'age_group', 'order_value_tier',
    'is_discounted', 'is_delivered', 'rating_bucket', 'delivery_speed'
]
clean_df[eng_cols].head(5)

---
## Stage 3 – Load

In [ ]:
from etl.load import load

outputs = load(clean_df, rejected_df)
for k, v in outputs.items():
    print(f'{k:25s}: {v}')

---
## Stage 4 – Analysis & Visualisation

In [ ]:
print('=== Key Metrics ===')
print(f"Total orders       : {len(clean_df):,}")
print(f"Total revenue      : ${clean_df['order_value'].sum():,.2f}")
print(f"Avg order value    : ${clean_df['order_value'].mean():,.2f}")
print(f"Avg customer rating: {clean_df['customer_rating'].mean():.2f}")
print(f"Delivery rate      : {clean_df['is_delivered'].mean()*100:.1f}%")
print(f"Discount rate      : {clean_df['is_discounted'].mean()*100:.1f}%")
print(f"Date range         : {clean_df['order_date'].min().date()} → {clean_df['order_date'].max().date()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = clean_df['order_status'].value_counts()
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('muted', len(status_counts)))
axes[0].set_title('Order Status Distribution')

cat_rev = (
    clean_df.groupby('product_category')['order_value']
    .sum().sort_values(ascending=True)
)
cat_rev.plot(kind='barh', ax=axes[1], color=sns.color_palette('muted', len(cat_rev)))
axes[1].set_title('Total Revenue by Category')
axes[1].set_xlabel('Revenue ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
monthly = (
    clean_df.groupby(['order_year', 'order_month'])
    .agg(orders=('order_id', 'count'), revenue=('order_value', 'sum'))
    .reset_index()
)
monthly['period'] = (
    monthly['order_year'].astype(str) + '-' +
    monthly['order_month'].astype(str).str.zfill(2)
)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(monthly['period'], monthly['orders'], color='steelblue', alpha=0.6, label='Orders')
ax2.plot(monthly['period'], monthly['revenue'], color='tomato', marker='o', linewidth=2, label='Revenue')
ax1.set_xlabel('Month')
ax1.set_ylabel('Order Count', color='steelblue')
ax2.set_ylabel('Revenue ($)', color='tomato')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.title('Monthly Orders & Revenue Trend')
plt.xticks(rotation=45, ha='right')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('../reports/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
delivery_dist = clean_df['delivery_speed'].value_counts()
rating_by_speed = clean_df.groupby('delivery_speed')['customer_rating'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
delivery_dist.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', 3))
axes[0].set_title('Delivery Speed Distribution')
axes[0].set_ylabel('Order Count')
axes[0].tick_params(axis='x', rotation=0)

rating_by_speed.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set1', 3))
axes[1].set_title('Avg Customer Rating by Delivery Speed')
axes[1].set_ylabel('Average Rating')
axes[1].set_ylim(0, 5.5)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/delivery_speed_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
conn = sqlite3.connect(config.DB_FILE)

print('=== Top 5 Categories by Revenue ===')
pd.read_sql("""
    SELECT product_category,
           COUNT(*) AS total_orders,
           ROUND(SUM(order_value), 2) AS total_revenue,
           ROUND(AVG(order_value), 2) AS avg_order_value,
           ROUND(AVG(customer_rating), 2) AS avg_rating
    FROM orders
    GROUP BY product_category
    ORDER BY total_revenue DESC
    LIMIT 5
""", conn)

In [ ]:
print('=== Pipeline Run Log ===')
pd.read_sql('SELECT * FROM pipeline_run_log ORDER BY run_ts DESC LIMIT 5', conn)

In [ ]:
conn.close()
print('Done. Outputs in data/processed/ and data/db/ecommerce.db')